In [0]:
%run ../utils/00_silver_utils

In [0]:
# Widget params; run_id comes from bronze via get_shared_run_id so this notebook's audit rows share bronze's run_id.
dbutils.widgets.text("silver_config_path", "/Volumes/cdr_silver/config/silver_config/silver_config.json")
dbutils.widgets.text("table_filter", "")

silver_config_path = dbutils.widgets.get("silver_config_path")
table_filter_raw = dbutils.widgets.get("table_filter").strip()
table_filter = {t.strip().lower() for t in table_filter_raw.split(",") if t.strip()} if table_filter_raw else None

dbutils.widgets.text("run_id", "")
run_id_param = dbutils.widgets.get("run_id").strip()
run_id = run_id_param if run_id_param else str(uuid.uuid4())
if not run_id_param:
    logger.warning(f"No run_id widget value supplied -- generating standalone run_id={run_id}")

logger.info(f"run_id              = {run_id}")
logger.info(f"silver_config_path  = {silver_config_path}")
logger.info(f"table_filter        = {table_filter or 'ALL TABLES FOR THIS NOTEBOOK'}")

In [0]:
silver_config = r"""{
  "patients": {
    "silver_notebook": "core_reference",
    "target_table": "patients",
    "merge_keys": ["subject_id"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "admissions": {
    "silver_notebook": "core_reference",
    "target_table": "admissions",
    "merge_keys": ["hadm_id"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "fill_missing", "args": {"column": "admission_location", "value": "unknown"}},
      {"fn": "fill_missing", "args": {"column": "discharge_location", "value": "unknown"}},
      {"fn": "fill_missing", "args": {"column": "language", "value": "others"}},
      {"fn": "fill_missing", "args": {"column": "marital_status", "value": "unknown"}},
      {"fn": "map_values_to_default", "args": {"column": "race", "values_to_map": ["Unknown", "Unable to Obtain", "Patient declined to answer"], "default_value": "Unknown"}}
    ]
  },
  "transfers": {
    "silver_notebook": "core_reference",
    "target_table": "transfers",
    "merge_keys": ["transfer_id"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "standardize_category", "args": {"column": "careunit"}}
    ]
  },
  "services": {
    "silver_notebook": "core_reference",
    "target_table": "services",
    "merge_keys": ["subject_id", "hadm_id", "transfertime", "prev_service", "curr_service"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "provider": {
    "silver_notebook": "core_reference",
    "target_table": "provider",
    "merge_keys": ["provider_id"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "caregiver": {
    "silver_notebook": "core_reference",
    "target_table": "caregiver",
    "merge_keys": ["caregiver_id"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "drgcodes": {
    "silver_notebook": "core_reference",
    "target_table": "drgcodes",
    "merge_keys": ["subject_id", "hadm_id", "drg_type", "drg_code"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "d_icd_diagnoses": {
    "silver_notebook": "core_reference",
    "target_table": "d_icd_diagnoses",
    "merge_keys": ["icd_code", "icd_version"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "rename_column", "args": {"old_name": "long_title", "new_name": "description"}}
    ]
  },
  "d_icd_procedures": {
    "silver_notebook": "core_reference",
    "target_table": "d_icd_procedures",
    "merge_keys": ["icd_code", "icd_version"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "rename_column", "args": {"old_name": "long_title", "new_name": "description"}}
    ]
  },
  "d_hcpcs": {
    "silver_notebook": "core_reference",
    "target_table": "d_hcpcs",
    "merge_keys": ["code"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "coalesce_and_drop", "args": {"col_a": "long_description", "col_b": "short_description", "new_col": "description"}}
    ]
  },
  "d_labitems": {
    "silver_notebook": "core_reference",
    "target_table": "d_labitems",
    "merge_keys": ["itemid"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "d_items": {
    "silver_notebook": "core_reference",
    "target_table": "d_items",
    "merge_keys": ["itemid"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "standardize_category", "args": {"column": "unitname"}}
    ]
  },
  "diagnoses_icd": {
    "silver_notebook": "clinical_events",
    "target_table": "icd_diagnoses",
    "merge_keys": ["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "procedures_icd": {
    "silver_notebook": "clinical_events",
    "target_table": "icd_procedures",
    "merge_keys": ["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "hcpcsevents": {
    "silver_notebook": "clinical_events",
    "target_table": "hcpcsevents",
    "merge_keys": ["subject_id", "hadm_id", "code", "seq_num"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "rename_column", "args": {"old_name": "hcpcs_cd", "new_name": "code"}},
      {"fn": "rename_column", "args": {"old_name": "short_description", "new_name": "description"}}
    ]
  },
  "omr": {
    "silver_notebook": "clinical_events",
    "target_table": "omr",
    "merge_keys": ["subject_id", "chartdate", "seq_num", "result_name"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "labevents": {
    "silver_notebook": "clinical_events",
    "target_table": "labevents",
    "merge_keys": ["labevent_id"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "drop_columns", "args": {"columns": ["storetime", "value"]}},
      {"fn": "fill_missing", "args": {"column": "flag", "value": "Unrecorded"}}
    ]
  },
  "microbiologyevents": {
    "silver_notebook": "clinical_events",
    "target_table": "microbiologyevents",
    "merge_keys": ["microevent_id"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "icustays": {
    "silver_notebook": "orders_meds_icu",
    "target_table": "icustays",
    "merge_keys": ["stay_id"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "poe": {
    "silver_notebook": "orders_meds_icu",
    "target_table": "poe",
    "merge_keys": ["poe_id"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "strip_html", "args": {"column": "order_subtype"}}
    ]
  },
  "poe_detail": {
    "silver_notebook": "orders_meds_icu",
    "target_table": "poe_detail",
    "merge_keys": ["poe_id", "field_name", "field_value"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "prescriptions": {
    "silver_notebook": "orders_meds_icu",
    "target_table": "prescriptions",
    "merge_keys": ["subject_id", "hadm_id", "pharmacy_id", "poe_id", "starttime", "stoptime", "drug_type", "drug"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "rename_column", "args": {"old_name": "formulary_drug_cd", "new_name": "formulary_drug_code"}}
    ]
  },
  "pharmacy": {
    "silver_notebook": "orders_meds_icu",
    "target_table": "pharmacy",
    "merge_keys": ["pharmacy_id"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "standardize_category", "args": {"column": "dispensation"}}
    ]
  },
  "emar": {
    "silver_notebook": "orders_meds_icu",
    "target_table": "emar",
    "merge_keys": ["emar_id"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "emar_detail": {
    "silver_notebook": "orders_meds_icu",
    "target_table": "emar_detail",
    "merge_keys": ["_natural_key"],
    "needs_surrogate_key": true,
    "cleaning_rules": [
      {"fn": "fill_missing", "args": {"column": "new_iv_bag_hung", "value": "Unrecorded"}},
      {"fn": "fill_missing", "args": {"column": "continued_infusion_in_other_location", "value": "Unrecorded"}},
      {"fn": "fill_missing", "args": {"column": "non_formulary_visual_verification", "value": "Unrecorded"}}
    ]
  },
  "procedureevents": {
    "silver_notebook": "orders_meds_icu",
    "target_table": "procedureevents",
    "merge_keys": ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "starttime", "endtime", "storetime", "orderid"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "standardize_category", "args": {"column": "location"}},
      {"fn": "cast_boolean", "args": {"column": "continueinnextdept"}}
    ]
  },
  "datetimeevents": {
    "silver_notebook": "orders_meds_icu",
    "target_table": "datetimeevents",
    "merge_keys": ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "charttime", "storetime"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "cast_boolean", "args": {"column": "warning"}},
      {"fn": "set_constant", "args": {"column": "valueuom", "value": "Date and Time"}}
    ]
  },
  "outputevents": {
    "silver_notebook": "orders_meds_icu",
    "target_table": "outputevents",
    "merge_keys": ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "charttime", "storetime"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "ingredientevents": {
    "silver_notebook": "orders_meds_icu",
    "target_table": "ingredientevents",
    "merge_keys": ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "starttime", "endtime", "storetime", "orderid"],
    "needs_surrogate_key": false,
    "cleaning_rules": []
  },
  "inputevents": {
    "silver_notebook": "orders_meds_icu",
    "target_table": "inputevents",
    "merge_keys": ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "starttime", "endtime", "storetime", "orderid"],
    "needs_surrogate_key": false,
    "cleaning_rules": [
      {"fn": "standardize_category", "args": {"column": "amountuom"}},
      {"fn": "standardize_category", "args": {"column": "ordercomponenttypedescription"}},
      {"fn": "cast_boolean", "args": {"column": "continueinnextdept"}}
    ]
  },
  "chartevents": {
    "silver_notebook": "orders_meds_icu",
    "target_table": "chartevents",
    "merge_keys": ["_natural_key"],
    "needs_surrogate_key": true,
    "cleaning_rules": [
      {"fn": "drop_columns", "args": {"columns": ["storetime"]}}
    ]
  }
}"""

dbutils.fs.put("/Volumes/cdr_silver/config/silver_config/silver_config.json",
               silver_config.strip(), overwrite=True)

import json
cfg = json.loads(silver_config)
print("tables:", len(cfg))
print(dbutils.fs.ls("/Volumes/cdr_silver/config/silver_config/"))

In [0]:
# Loads silver_config.json and filters it down to the tables tagged silver_notebook == "core_reference".
with open(silver_config_path, "r") as f:
    SILVER_CONFIG = json.load(f)

LAYER = "core_reference"
tables_for_this_notebook = [
    t for t, cfg in SILVER_CONFIG.items() if cfg["silver_notebook"] == LAYER
]
if table_filter:
    tables_for_this_notebook = [t for t in tables_for_this_notebook if t in table_filter]

logger.info(f"Processing {len(tables_for_this_notebook)} table(s) for layer '{LAYER}': {tables_for_this_notebook}")

ensure_silver_audit_table(spark)

In [0]:
# Bespoke transform for admissions: fixes ~175 rows where admittime is after dischtime, silently (no audit flag column).
def clean_admissions(df):
    # ~175 rows have admittime > dischtime, which should never happen. Fix (no flag column -- confirmed no flag needed):
    #   1) if edregtime and edouttime both exist and edregtime < edouttime: admittime = edregtime, dischtime = edouttime
    #   2) otherwise: keep admittime as-is, set dischtime = admittime + 1 hour
    bad_row = col("admittime") > col("dischtime")
    ed_valid = col("edregtime").isNotNull() & col("edouttime").isNotNull() & (col("edregtime") < col("edouttime"))

    df = df.withColumn(
        "dischtime",
        when(bad_row & ed_valid, col("edouttime"))
        .when(bad_row & ~ed_valid, col("admittime") + expr("INTERVAL 1 HOURS"))
        .otherwise(col("dischtime"))
    )
    df = df.withColumn(
        "admittime",
        when(bad_row & ed_valid, col("edregtime")).otherwise(col("admittime"))
    )
    return df

In [0]:
# Bespoke transform for drgcodes: makes description single-valued per (drg_type, drg_code) in place -- no separate dimension table.
from pyspark.sql import Window
from pyspark.sql.functions import first as _first

def clean_drgcodes(df):
    # Each (drg_type, drg_code) must map to a single description -- no separate dim table (confirmed).
    # Tiebreak rule (not specified beyond "prefer non-null" -- chosen here for determinism across reruns):
    # alphabetically-first non-null description per (drg_type, drg_code) group.
    w = Window.partitionBy("drg_type", "drg_code").orderBy(col("description").asc_nulls_last())
    return df.withColumn("description", _first(col("description"), ignorenulls=True).over(w))

In [0]:
# Maps each table needing bespoke logic (not expressible as a generic cleaning_rules entry) to its function.
SPECIAL_TRANSFORMS = {
    "admissions": clean_admissions,
    "drgcodes": clean_drgcodes
}

In [0]:
# Runs every table for this layer: read bronze, clean, write/merge to silver, collect an audit row, then write the audit log.
audit_rows = []
success_count = 0
failure_count = 0

for table_name in tables_for_this_notebook:
    audit_row = process_table(
        spark, table_name, SILVER_CONFIG, run_id, layer=LAYER,
        special_transform=SPECIAL_TRANSFORMS.get(table_name),
    )
    audit_rows.append(audit_row)
    if audit_row.status == "SUCCESS":
        success_count += 1
    else:
        failure_count += 1

write_audit_rows(spark, audit_rows)

logger.info(
    f"Layer '{LAYER}' complete | run_id={run_id} | {success_count} succeeded | "
    f"{failure_count} failed | {len(tables_for_this_notebook)} total tables"
)
if failure_count > 0:
    logger.warning(f"{failure_count} table(s) failed -- check {SILVER_AUDIT_TABLE} (run_id = {run_id})")